# Оптимизированная система обработки и анализа логов веб-сервера

##### Постановка задачи:
У нас есть логи веб-сервера (Nginx/Apache), которые нужно:
1. Загрузить и очистить (обработка ошибок, парсинг)
2. Отсортировать по разным критериям (время, IP, статус, размер ответа)
3. Проанализировать (статистика, аномалии, паттерны)
4. Визуализировать результаты:

##### Что это даст:
* Практическая задача (реальная проблема DevOps/аналитиков)
* Большие данные (логи легко генерировать/скачать)
* Естественное использование изученных технологий
* Три реализации (однопоточная, многопоточная, гибридная)
* Сравнение алгоритмов на реальных данных

In [25]:
# Импортируем необходимые компоненты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import re
import hashlib
import random
import time
import json
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Any, Optional, Generator
import concurrent.futures
import threading
from functools import reduce
from tqdm import tqdm
import os

In [26]:
class SimpleLogGenerator:
    """Генератор веб-логов"""

    def __init__(self, seed=42):
        random.seed(seed)
        np.random.seed(seed)

        self.resources = ['/index.html', '/about', '/contact', '/api/data',
                         '/products', '/login', '/logout', '/register',
                         '/css/style.css', '/js/app.js', '/images/logo.png']

        self.status_codes = {
            200: 0.7,   # OK
            301: 0.05,  # Moved Permanently
            302: 0.05,  # Found
            404: 0.1,   # Not Found
            500: 0.05,  # Internal Server Error
            403: 0.05   # Forbidden
        }

        # Список IP адресов
        self.ip_pool = [f"192.168.{i}.{j}" for i in range(1, 5) for j in range(1, 50)]

        # Список User Agents
        self.user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)',
            'Mozilla/5.0 (X11; Linux x86_64)',
            'Mozilla/5.0 (iPhone; CPU iPhone OS 14_0)',
            'Googlebot/2.1 (+http://www.google.com/bot.html)'
        ]

    def generate_simple_log_entry(self) -> str:
        """Генерирует простую запись лога"""
        ip = random.choice(self.ip_pool)
        timestamp = datetime.now() - timedelta(seconds=random.randint(0, 2592000))
        timestamp_str = timestamp.strftime('[%d/%b/%Y:%H:%M:%S %z]')

        method = random.choice(['GET', 'POST'])
        resource = random.choice(self.resources)

        status = random.choices(
            list(self.status_codes.keys()),
            weights=list(self.status_codes.values())
        )[0]

        if status == 200:
            bytes_sent = random.randint(500, 50000)
        elif status == 404:
            bytes_sent = 0
        else:
            bytes_sent = random.randint(0, 1000)

        referrer = random.choice(['-', 'https://google.com', 'https://www.apple.com'])
        user_agent = random.choice(self.user_agents)

        # Упрощенный формат: IP - - [timestamp] "method resource" status bytes
        return f'{ip} - - {timestamp_str} "{method} {resource} HTTP/1.1" {status} {bytes_sent} "{referrer}" "{user_agent}"'

    def generate_logs(self, num_entries: int = 1000) -> List[str]:
        """Генерирует указанное количество записей лога"""
        print(f"Генерация {num_entries:,} записей логов...")
        return [self.generate_simple_log_entry() for _ in tqdm(range(num_entries))]

    def save_logs_to_file(self, logs: List[str], filename: str = "web_server.log"):
        """Сохраняет логи в файл"""
        with open(filename, 'w') as f:
            for log in logs:
                f.write(log + '\n')
        print(f"Логи сохранены в файл: {filename}")
        return filename


In [27]:
# Парсер логов
class LogEntry:
    """Класс для представления одной записи лога"""

    def __init__(self, raw_log: str):
        self.raw_log = raw_log
        self.ip = ""
        self.timestamp = ""
        self.method = ""
        self.resource = ""
        self.protocol = ""
        self.status = 0
        self.bytes_sent = 0
        self.referrer = ""
        self.user_agent = ""
        self.parsed = False
        self.error = None

        self._parse_simple_log()

    def _parse_simple_log(self):
        """Парсинг логов"""
        try:
            pattern = r'^(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) (\S+)" (\d+) (\d+) "([^"]*)" "([^"]*)"'
            match = re.match(pattern, self.raw_log.strip())

            if match:
                self.ip = match.group(1)
                self.timestamp = match.group(2)
                self.method = match.group(3)
                self.resource = match.group(4)
                self.protocol = match.group(5)
                self.status = int(match.group(6))
                self.bytes_sent = int(match.group(7))
                self.referrer = match.group(8)
                self.user_agent = match.group(9)
                self.parsed = True
            else:
                alt_pattern = r'^(\S+).*\[([^\]]+)\].*"(\S+) (\S+).*" (\d+) (\d+)'
                alt_match = re.match(alt_pattern, self.raw_log.strip())
                if alt_match:
                    self.ip = alt_match.group(1)
                    self.timestamp = alt_match.group(2)
                    self.method = alt_match.group(3)
                    self.resource = alt_match.group(4)
                    self.status = int(alt_match.group(5))
                    self.bytes_sent = int(alt_match.group(6))
                    self.parsed = True
                else:
                    self.error = "Failed to parse log entry"

        except Exception as e:
            self.error = str(e)

    def __str__(self) -> str:
        """Строковое представление для пользователя"""
        return f"{self.ip} - {self.method} {self.resource} - {self.status}"

    def __repr__(self) -> str:
        """Представление для разработчика"""
        return f"LogEntry(ip='{self.ip}', method='{self.method}', resource='{self.resource}', status={self.status})"

    def to_dict(self) -> Dict:
        """Преобразует запись в словарь"""
        return {
            'ip': self.ip,
            'timestamp': self.timestamp,
            'method': self.method,
            'resource': self.resource,
            'protocol': self.protocol,
            'status': self.status,
            'bytes_sent': self.bytes_sent,
            'referrer': self.referrer,
            'user_agent': self.user_agent
        }

In [28]:
def timer_decorator(func):
    """Декоратор для измерения времени выполнения функции"""
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"{func.__name__} выполнена за {end_time - start_time:.4f} секунд")
        return result
    return wrapper

In [29]:
class LogProcessor:
    """
    Класс для обработки и анализа веб-логов
    """

    def __init__(self, name: str = "LogProcessor"):
        self.name = name
        self.log_entries: List[LogEntry] = []
        self.parsing_stats = {'total': 0, 'success': 0, 'failed': 0}
        self.analysis_results = {}

    def __str__(self) -> str:
        return f"LogProcessor '{self.name}' с {len(self.log_entries)} записями"

    def __repr__(self) -> str:
        return f"LogProcessor(name='{self.name}', entries={len(self.log_entries)})"

    def __len__(self) -> int:
        return len(self.log_entries)

    def __getitem__(self, idx: int) -> LogEntry:
        if 0 <= idx < len(self.log_entries):
            return self.log_entries[idx]
        raise IndexError(f"Индекс {idx} вне диапазона (0-{len(self.log_entries)-1})")

    def __iter__(self):
        return iter(self.log_entries)

    def __contains__(self, ip: str) -> bool:
        return any(entry.ip == ip for entry in self.log_entries if entry.parsed)

    @timer_decorator
    def load_logs_from_list(self, logs: List[str]) -> 'LogProcessor':
        """Загружает логи из списка строк"""
        print(f"Загрузка {len(logs):,} записей логов...")

        self.log_entries = []
        self.parsing_stats = {'total': 0, 'success': 0, 'failed': 0}

        for log in tqdm(logs, desc="Парсинг логов"):
            self.parsing_stats['total'] += 1
            entry = LogEntry(log)

            if entry.parsed:
                self.log_entries.append(entry)
                self.parsing_stats['success'] += 1
            else:
                self.parsing_stats['failed'] += 1

        print(f"Успешно распаршено: {self.parsing_stats['success']:,} / {self.parsing_stats['total']:,}")

        if self.parsing_stats['success'] == 0:
            print("Внимание: не удалось распарсить ни одной записи!")
            print("Пример неудачной записи:", logs[0] if logs else "нет данных")

        return self

    # реализация map, reduce, filter
    def filter_by_status(self, status_code: int) -> List[LogEntry]:
        """Фильтрует логи по статус коду"""
        return list(filter(lambda entry: entry.parsed and entry.status == status_code, self.log_entries))

    def filter_by_ip(self, ip: str) -> List[LogEntry]:
        """Фильтрует логи по IP адресу"""
        return list(filter(lambda entry: entry.parsed and entry.ip == ip, self.log_entries))

    def map_to_status_list(self) -> List[int]:
        """Преобразует логи в список статусов"""
        return list(map(lambda entry: entry.status if entry.parsed else 0, self.log_entries))

    def reduce_to_total_bytes(self) -> int:
        """Суммирует общее количество переданных байт"""
        return reduce(lambda total, entry: total + (entry.bytes_sent if entry.parsed else 0),
                     self.log_entries, 0)

    # Генератор для обработки по частям
    def log_generator(self, chunk_size: int = 100) -> Generator[List[LogEntry], None, None]:
        """Генератор для обработки логов по частям"""
        for i in range(0, len(self.log_entries), chunk_size):
            yield self.log_entries[i:i + chunk_size]

    # Многопоточная обработка
    def process_chunk(self, chunk: List[LogEntry]) -> Dict:
        """Обрабатывает чанк логов"""
        chunk_stats = {
            'status_counts': defaultdict(int),
            'total_bytes': 0,
            'count': 0
        }

        for entry in chunk:
            if entry.parsed:
                chunk_stats['status_counts'][entry.status] += 1
                chunk_stats['total_bytes'] += entry.bytes_sent
                chunk_stats['count'] += 1

        return chunk_stats

    @timer_decorator
    def parallel_analysis(self, num_threads: int = 4) -> Dict:
        """Параллельный анализ логов"""
        print(f"Параллельный анализ с {num_threads} потоками...")

        if not self.log_entries:
            return {}

        chunk_size = len(self.log_entries) // num_threads
        chunks = [self.log_entries[i:i + chunk_size] for i in range(0, len(self.log_entries), chunk_size)]
        chunks = chunks[:num_threads]

        results = {
            'status_counts': defaultdict(int),
            'total_bytes': 0,
            'total_requests': 0
        }

        with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
            future_to_chunk = {executor.submit(self.process_chunk, chunk): i for i, chunk in enumerate(chunks)}

            for future in concurrent.futures.as_completed(future_to_chunk):
                chunk_result = future.result()

                for status, count in chunk_result['status_counts'].items():
                    results['status_counts'][status] += count

                results['total_bytes'] += chunk_result['total_bytes']
                results['total_requests'] += chunk_result['count']

        results['status_counts'] = dict(results['status_counts'])

        self.analysis_results = results
        return results

    # Анализ однопоточный
    @timer_decorator
    def single_thread_analysis(self) -> Dict:
        """Однопоточный анализ логов"""
        print("Однопоточный анализ...")

        if not self.log_entries:
            return {}

        results = {
            'status_counts': defaultdict(int),
            'total_bytes': 0,
            'total_requests': 0
        }

        for entry in tqdm(self.log_entries, desc="Анализ"):
            if entry.parsed:
                results['status_counts'][entry.status] += 1
                results['total_bytes'] += entry.bytes_sent
                results['total_requests'] += 1

        results['status_counts'] = dict(results['status_counts'])
        return results

    # Анадитические методы
    def analyze_requests(self) -> Dict[str, Any]:
        """Анализирует запросы и возвращает статистику"""
        if not self.log_entries:
            return {}

        total_requests = len([e for e in self.log_entries if e.parsed])
        successful_requests = len([e for e in self.log_entries if e.parsed and 200 <= e.status < 300])
        error_requests = len([e for e in self.log_entries if e.parsed and e.status >= 400])

        resource_counter = Counter()
        for entry in self.log_entries:
            if entry.parsed:
                resource_counter[entry.resource] += 1

        most_popular_resource = resource_counter.most_common(1)[0] if resource_counter else ("нет данных", 0)

        ip_counter = Counter()
        for entry in self.log_entries:
            if entry.parsed:
                ip_counter[entry.ip] += 1

        most_active_ip = ip_counter.most_common(1)[0] if ip_counter else ("нет данных", 0)

        return {
            'total_requests': total_requests,
            'successful_requests': successful_requests,
            'error_requests': error_requests,
            'success_rate': (successful_requests / total_requests * 100) if total_requests > 0 else 0,
            'error_rate': (error_requests / total_requests * 100) if total_requests > 0 else 0,
            'most_popular_resource': most_popular_resource[0],
            'most_popular_resource_count': most_popular_resource[1],
            'most_active_ip': most_active_ip[0],
            'most_active_ip_count': most_active_ip[1]
        }

    def find_top_n_ips(self, n: int = 5) -> List[Tuple[str, int]]:
        """Находит топ-N IP адресов по количеству запросов"""
        ip_counter = Counter()
        for entry in self.log_entries:
            if entry.parsed:
                ip_counter[entry.ip] += 1

        return ip_counter.most_common(n)

    def find_top_n_resources(self, n: int = 5) -> List[Tuple[str, int]]:
        """Находит топ-N самых запрашиваемых ресурсов"""
        resource_counter = Counter()
        for entry in self.log_entries:
            if entry.parsed:
                resource_counter[entry.resource] += 1

        return resource_counter.most_common(n)

In [30]:
class PerformanceComparator:
    """Сравнивает производительность разных методов анализа"""

    def __init__(self):
        self.results = {}

    def compare_methods(self, processor: LogProcessor, data_size: int) -> Dict:
        """Сравнивает однопоточный и многопоточный методы"""
        print(f"\nСравнение методов для {data_size:,} записей:")
        print("="*50)

        print("\n1. ОДНОПОТОЧНЫЙ МЕТОД:")
        start_time = time.time()
        single_result = processor.single_thread_analysis()
        single_time = time.time() - start_time

        print("\n2. МНОГОПОТОЧНЫЙ МЕТОД (4 потока):")
        start_time = time.time()
        multi_result = processor.parallel_analysis(num_threads=4)
        multi_time = time.time() - start_time

        print("\n3. MAP/REDUCE МЕТОД:")
        start_time = time.time()

        parsed_entries = list(filter(lambda e: e.parsed, processor.log_entries))

        mapped_data = list(map(lambda e: {
            'status': e.status,
            'bytes': e.bytes_sent,
            'ip': e.ip,
            'resource': e.resource
        }, parsed_entries))

        status_counts = reduce(
            lambda acc, item: {**acc, item['status']: acc.get(item['status'], 0) + 1},
            mapped_data,
            {}
        )

        total_bytes = reduce(
            lambda acc, item: acc + item['bytes'],
            mapped_data,
            0
        )

        map_reduce_time = time.time() - start_time

        comparison = {
            'single_thread': {
                'time': single_time,
                'requests_processed': single_result.get('total_requests', 0),
                'total_bytes': single_result.get('total_bytes', 0)
            },
            'multi_thread': {
                'time': multi_time,
                'requests_processed': multi_result.get('total_requests', 0),
                'total_bytes': multi_result.get('total_bytes', 0)
            },
            'map_reduce': {
                'time': map_reduce_time,
                'requests_processed': len(parsed_entries),
                'total_bytes': total_bytes
            },
            'data_size': data_size
        }

        if multi_time > 0:
            comparison['speedup_multi_vs_single'] = single_time / multi_time

        if map_reduce_time > 0:
            comparison['speedup_mapreduce_vs_single'] = single_time / map_reduce_time

        self.results[data_size] = comparison
        self._print_comparison(comparison)

        return comparison

    def _print_comparison(self, comparison: Dict):
        """Выводит результаты сравнения"""
        print("\n" + "="*50)
        print("РЕЗУЛЬТАТЫ СРАВНЕНИЯ:")
        print("="*50)

        methods = ['single_thread', 'multi_thread', 'map_reduce']
        method_names = ['Однопоточный', 'Многопоточный', 'Map/Reduce']

        for method, name in zip(methods, method_names):
            data = comparison[method]
            print(f"\n{name}:")
            print(f"  Время: {data['time']:.4f} сек")
            print(f"  Обработано запросов: {data['requests_processed']:,}")
            print(f"  Всего байт: {data['total_bytes']:,}")

        fastest_method = min(methods, key=lambda m: comparison[m]['time'])
        fastest_name = method_names[methods.index(fastest_method)]

        print(f"\nСамый быстрый метод: {fastest_name} ({comparison[fastest_method]['time']:.4f} сек)")

        if 'speedup_multi_vs_single' in comparison:
            print(f"Ускорение многопоточного метода: {comparison['speedup_multi_vs_single']:.2f}x")

        if 'speedup_mapreduce_vs_single' in comparison:
            print(f"Ускорение Map/Reduce метода: {comparison['speedup_mapreduce_vs_single']:.2f}x")

    def visualize_comparison(self):
        """Визуализирует результаты сравнения"""
        if not self.results:
            print("Нет данных для визуализации")
            return

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        sizes = sorted(self.results.keys())

        single_times = [self.results[s]['single_thread']['time'] for s in sizes]
        multi_times = [self.results[s]['multi_thread']['time'] for s in sizes]
        mapreduce_times = [self.results[s]['map_reduce']['time'] for s in sizes]

        axes[0, 0].plot(sizes, single_times, marker='o', label='Однопоточный', linewidth=2)
        axes[0, 0].plot(sizes, multi_times, marker='s', label='Многопоточный', linewidth=2)
        axes[0, 0].plot(sizes, mapreduce_times, marker='^', label='Map/Reduce', linewidth=2)
        axes[0, 0].set_title('Время выполнения методов')
        axes[0, 0].set_xlabel('Количество записей')
        axes[0, 0].set_ylabel('Время (секунды)')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        speedups = []
        for s in sizes:
            if self.results[s]['single_thread']['time'] > 0:
                speedup = self.results[s]['single_thread']['time'] / self.results[s]['multi_thread']['time']
                speedups.append(speedup)

        axes[0, 1].bar(range(len(sizes)), speedups, color='lightcoral')
        axes[0, 1].set_title('Ускорение многопоточного метода')
        axes[0, 1].set_xlabel('Размер данных')
        axes[0, 1].set_ylabel('Коэффициент ускорения')
        axes[0, 1].set_xticks(range(len(sizes)))
        axes[0, 1].set_xticklabels([f"{s:,}" for s in sizes])
        axes[0, 1].axhline(y=1, color='r', linestyle='--', alpha=0.5)

        reqs_per_sec_single = [self.results[s]['single_thread']['requests_processed'] /
                              self.results[s]['single_thread']['time'] if self.results[s]['single_thread']['time'] > 0 else 0
                              for s in sizes]
        reqs_per_sec_multi = [self.results[s]['multi_thread']['requests_processed'] /
                             self.results[s]['multi_thread']['time'] if self.results[s]['multi_thread']['time'] > 0 else 0
                             for s in sizes]

        axes[1, 0].bar([s - 500 for s in sizes], reqs_per_sec_single, width=500, label='Однопоточный', alpha=0.7)
        axes[1, 0].bar(sizes, reqs_per_sec_multi, width=500, label='Многопоточный', alpha=0.7)
        axes[1, 0].set_title('Производительность (запросов в секунду)')
        axes[1, 0].set_xlabel('Количество записей')
        axes[1, 0].set_ylabel('Запросов/сек')
        axes[1, 0].legend()
        axes[1, 0].set_xticks(sizes)
        axes[1, 0].set_xticklabels([f"{s:,}" for s in sizes])

        bytes_processed = [self.results[s]['single_thread']['total_bytes'] for s in sizes]
        axes[1, 1].plot(sizes, bytes_processed, marker='o', color='green', linewidth=2)
        axes[1, 1].set_title('Объем обработанных данных')
        axes[1, 1].set_xlabel('Количество записей')
        axes[1, 1].set_ylabel('Всего байт')
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

In [31]:
def simple_demo():
    """Простая демонстрация работы системы"""
    print("="*70)
    print("СИСТЕМА АНАЛИЗА ВЕБ-ЛОГОВ")
    print("="*70)

    print("\n1. ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ")
    generator = SimpleLogGenerator(seed=42)
    test_logs = generator.generate_logs(1000)
    print(f"Сгенерировано {len(test_logs):,} записей логов")
    print(f"Пример записи: {test_logs[0]}")

    print("\n2. СОЗДАНИЕ ПРОЦЕССОРА ЛОГОВ")
    processor = LogProcessor("Демо-процессор")

    processor.load_logs_from_list(test_logs)

    print("\n3. ДЕМОНСТРАЦИЯ ПЕРЕОПРЕДЕЛЕННЫХ МЕТОДОВ КЛАССА:")
    print(f"  str:  {str(processor)}")
    print(f"  repr: {repr(processor)}")
    print(f"  len:  {len(processor)}")

    if len(processor) > 0:
        print(f"  processor[0]: {processor[0]}")
        print(f"  Содержит IP {processor[0].ip}? {processor[0].ip in processor}")

    print("\n4. ДЕМОНСТРАЦИЯ MAP/REDUCE/FILTER:")

    successful_logs = processor.filter_by_status(200)
    print(f"  Успешных запросов (200): {len(successful_logs):,}")

    status_list = processor.map_to_status_list()
    unique_statuses = set(status_list)
    print(f"  Уникальных статусов: {len(unique_statuses)}")

    total_bytes = processor.reduce_to_total_bytes()
    print(f"  Общий объем трафика: {total_bytes:,} байт ({total_bytes/1024/1024:.2f} MB)")

    print("\n5. АНАЛИЗ ДАННЫХ:")
    analysis = processor.analyze_requests()
    print(f"  Всего запросов: {analysis['total_requests']:,}")
    print(f"  Успешных запросов: {analysis['successful_requests']:,} ({analysis['success_rate']:.1f}%)")
    print(f"  Ошибочных запросов: {analysis['error_requests']:,} ({analysis['error_rate']:.1f}%)")
    print(f"  Самый популярный ресурс: {analysis['most_popular_resource']} ({analysis['most_popular_resource_count']} запросов)")
    print(f"  Самый активный IP: {analysis['most_active_ip']} ({analysis['most_active_ip_count']} запросов)")

    top_ips = processor.find_top_n_ips(3)
    print(f"  Топ-3 IP адресов:")
    for ip, count in top_ips:
        print(f"    {ip}: {count} запросов")

    top_resources = processor.find_top_n_resources(3)
    print(f"  Топ-3 ресурсов:")
    for resource, count in top_resources:
        print(f"    {resource}: {count} запросов")

    return processor

def performance_comparison_demo():
    """Демонстрация сравнения производительности"""
    print("\n" + "="*70)
    print("СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ РАЗНЫХ МЕТОДОВ")
    print("="*70)

    comparator = PerformanceComparator()

    sizes = [500, 2000, 5000, 10000]

    for size in sizes:
        print(f"\n{'='*60}")
        print(f"ТЕСТИРОВАНИЕ НА {size:,} ЗАПИСЯХ")
        print(f"{'='*60}")

        generator = SimpleLogGenerator(seed=size)
        logs = generator.generate_logs(size)

        processor = LogProcessor(f"Processor_{size}")
        processor.load_logs_from_list(logs)

        comparator.compare_methods(processor, size)

    comparator.visualize_comparison()

    print("\n" + "="*70)
    print("СРАВНЕНИЕ ЗАВЕРШЕНО!")
    print("="*70)

def advanced_demo():
    """Расширенная демонстрация с обработкой ошибок и генераторами"""
    print("="*70)
    print("РАСШИРЕННАЯ ДЕМОНСТРАЦИЯ")
    print("="*70)

    generator = SimpleLogGenerator(seed=123)
    large_logs = generator.generate_logs(5000)

    processor = LogProcessor("Расширенный процессор")

    try:
        processor.load_logs_from_list(large_logs)

        print("\nОБРАБОТКА ЛОГОВ С ИСПОЛЬЗОВАНИЕМ ГЕНЕРАТОРА:")

        total_chunks = 0
        total_entries = 0

        for chunk in processor.log_generator(chunk_size=100):
            total_chunks += 1
            total_entries += len(chunk)
            print(f"  Чанк {total_chunks}: {len(chunk)} записей")

            if total_chunks >= 5:
                print(f"  ... и еще {len(processor) - total_entries} записей в следующих чанках")
                break

        print("\nАНАЛИЗ С ИСПОЛЬЗОВАНИЕМ MATCH CASE:")
        status_summary = defaultdict(int)
        for entry in processor.log_entries[:50]:
            if entry.parsed:
                match entry.status:
                    case 200:
                        status_summary['Успешно'] += 1
                    case 301 | 302:
                        status_summary['Перенаправление'] += 1
                    case 404:
                        status_summary['Не найдено'] += 1
                    case 500:
                        status_summary['Ошибка сервера'] += 1
                    case 403:
                        status_summary['Запрещено'] += 1
                    case _:
                        status_summary['Другое'] += 1

        print("  Распределение статусов (первые 50 записей):")
        for category, count in status_summary.items():
            print(f"    {category}: {count}")

    except Exception as e:
        print(f"Произошла ошибка: {e}")

    finally:
        print("\nДЕМОНСТРАЦИЯ ЗАВЕРШЕНА (блок finally выполнен)")

In [ ]:
if __name__ == "__main__":
    # Простая демонстрация
    processor = simple_demo()

    # Расширенная демонстрация
    advanced_demo()

    # Сравнение производительности
    # performance_comparison_demo()

    print("\n" + "="*70)
    print("✅ ВСЕ ДЕМОНСТРАЦИИ УСПЕШНО ВЫПОЛНЕНЫ!")
    print("="*70)

ЗАПУСК СИСТЕМЫ АНАЛИЗА ВЕБ-ЛОГОВ

СИСТЕМА АНАЛИЗА ВЕБ-ЛОГОВ

1. ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ
Генерация 1,000 записей логов...


100%|██████████| 1000/1000 [00:00<00:00, 45460.30it/s]


Сгенерировано 1,000 записей логов
Пример записи: 192.168.4.17 - - [12/Dec/2025:10:50:17 ] "GET /products HTTP/1.1" 200 9644 "https://www.apple.com" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"

2. СОЗДАНИЕ ПРОЦЕССОРА ЛОГОВ
Загрузка 1,000 записей логов...


Парсинг логов: 100%|██████████| 1000/1000 [00:00<00:00, 142368.01it/s]


Успешно распаршено: 1,000 / 1,000
load_logs_from_list выполнена за 0.0110 секунд

3. ДЕМОНСТРАЦИЯ ПЕРЕОПРЕДЕЛЕННЫХ МЕТОДОВ КЛАССА:
  str:  LogProcessor 'Демо-процессор' с 1000 записями
  repr: LogProcessor(name='Демо-процессор', entries=1000)
  len:  1000
  processor[0]: 192.168.4.17 - GET /products - 200
  Содержит IP 192.168.4.17? True

4. ДЕМОНСТРАЦИЯ MAP/REDUCE/FILTER:
  Успешных запросов (200): 702
  Уникальных статусов: 6
  Общий объем трафика: 18,170,768 байт (17.33 MB)

5. АНАЛИЗ ДАННЫХ:
  Всего запросов: 1,000
  Успешных запросов: 702 (70.2%)
  Ошибочных запросов: 202 (20.2%)
  Самый популярный ресурс: /about (104 запросов)
  Самый активный IP: 192.168.2.8 (13 запросов)
  Топ-3 IP адресов:
    192.168.2.8: 13 запросов
    192.168.2.22: 12 запросов
    192.168.2.49: 11 запросов
  Топ-3 ресурсов:
    /about: 104 запросов
    /register: 98 запросов
    /products: 97 запросов
РАСШИРЕННАЯ ДЕМОНСТРАЦИЯ
Генерация 5,000 записей логов...


100%|██████████| 5000/5000 [00:00<00:00, 58823.83it/s]


Загрузка 5,000 записей логов...


Парсинг логов: 100%|██████████| 5000/5000 [00:00<00:00, 207972.39it/s]

Успешно распаршено: 5,000 / 5,000
load_logs_from_list выполнена за 0.0270 секунд

ОБРАБОТКА ЛОГОВ С ИСПОЛЬЗОВАНИЕМ ГЕНЕРАТОРА:
  Чанк 1: 100 записей
  Чанк 2: 100 записей
  Чанк 3: 100 записей
  Чанк 4: 100 записей
  Чанк 5: 100 записей
  ... и еще 4500 записей в следующих чанках

АНАЛИЗ С ИСПОЛЬЗОВАНИЕМ MATCH CASE:
  Распределение статусов (первые 50 записей):
    Успешно: 38
    Ошибка сервера: 5
    Перенаправление: 3
    Не найдено: 3
    Запрещено: 1

ДЕМОНСТРАЦИЯ ЗАВЕРШЕНА (блок finally выполнен)

✅ ВСЕ ДЕМОНСТРАЦИИ УСПЕШНО ВЫПОЛНЕНЫ!
